# Ders 01 - AI Ajanlarına Giriş

**Başlangıç Seviyesi AI Ajanları** kursunun ilk dersine hoş geldiniz!

**AI ajanı**, büyük bir dil modelini (LLM) mantık motoru olarak kullanan ve gerçek dünyada *eylemler* gerçekleştirebilen — API'leri çağırma, veri tabanlarını sorgulama veya kod çalıştırma gibi — bir programdır ve kullanıcının adına bir hedefi gerçekleştirebilir.

Bu not defterinde ilk ajanınızı oluşturacaksınız: tatil yerleri öneren bir **Seyahat Ajanı**. Yol boyunca şunları öğreneceksiniz:

1. **Microsoft Agent Framework** kullanarak Azure AI Foundry Agent Servisi'ne bağlanmayı.
2. Ajanınıza çağırabileceği bir **araç** vermeyi — basit bir Python fonksiyonu.
3. Ajanı çalıştırmayı ve yanıtını incelemeyi.
4. Ajanın yanıtını token token akıtmayı.


## Kurulum

Bu defteri çalıştırmadan önce, şunlara sahip olduğunuzdan emin olun:

1. **Bir Azure AI Foundry projesi** ve üzerinde konuşma modeli dağıtılmış (örneğin `gpt-4o-mini`).
2. **Azure CLI ile giriş yapılmış** — terminalinizde `az login` komutunu çalıştırın.
3. **Gerekli ortam değişkenleri ayarlanmış:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry proje uç noktanız.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — dağıtılan modelinizin adı.

Aşağıdaki hücre, ihtiyacınız olan Python paketlerini yükler.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## İlk Ajanınızı Oluşturmak

Bir ajanın iki şeye ihtiyacı vardır:

- Ona *kim olduğunu* ve *nasıl davranacağını* söyleyen **Talimatlar** (bir sistem istemi).
- Ajansın bilgi almak veya işlem yapmak için çağırabileceği `@tool` ile işaretlenmiş Python fonksiyonları — **Araçlar**.

Aşağıda popüler tatil yerlerinin bir listesini döndüren basit bir araç tanımlıyoruz. Kullanıcı seyahat tavsiyeleri istediğinde ajan bu aracı kullanacak.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Akış Yanıtları

Daha etkileşimli bir deneyim için, ajan yanıtını **akış halinde** alabilirsiniz. Tam yanıtı beklemek yerine, ajan metin parçalarını oluşturdukça iletir. Bu, çıktıyı gerçek zamanlı göstermek istediğiniz sohbet arayüzlerinde özellikle faydalıdır.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Özet

Bu derste şunları öğrendiniz:

- `AzureAIProjectAgentProvider` aracılığıyla Azure AI Foundry Agent Servisine bağlanan bir **provider oluşturmayı**.
- Ajanın Python fonksiyonlarınızı çağırabilmesi için `@tool` dekoratörünü kullanarak bir **araç tanımlamayı**.
- Kullanıcı mesajı ile **ajani çalıştırıp yanıtını yazdırmayı**.
- Gerçek zamanlı çıktı için **yanıtları akış halinde almayı**.

Bir sonraki derste ajanik çerçeveleri daha derinlemesine inceleyecek ve ajanlara daha güçlü araçlar ve çok adımlı akıl yürütme yetenekleri vermeyi öğreneceğiz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba göstersek de, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Önemli bilgiler için profesyonel insan çevirisi tavsiye edilir. Bu çevirinin kullanımı sonucu oluşabilecek yanlış anlamalar veya yorum hatalarından sorumlu tutulamayız.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
